# 10 · Time series forecasting and backtesting

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Create lag features; avoid future leakage; compare a forecast with a seasonal baseline.


## Step 1 — Build a daily series

Time order matters. Tomorrow must never help predict yesterday.


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
rng = np.random.default_rng(42)
t = np.arange(420)
y = 50 + 0.03*t + 8*np.sin(2*np.pi*t/7) + rng.normal(0,1,len(t))
lags = [1,7,14]
indices = np.arange(max(lags),len(y))
X = np.array([[y[i-lag] for lag in lags] + [float(i)] for i in indices])
target = y[indices]
cut = int(len(target)*0.8)
print('Training last index:',indices[cut-1],'test first:',indices[cut])


**Check your result:** Every feature references only an earlier time step.


## Step 2 — Backtest on the past

Expanding-window folds simulate one-step-ahead predictions when the previous actual observations are available.


In [ ]:
fold_mae=[]
for tr,va in TimeSeriesSplit(n_splits=3).split(X[:cut]):
    model=Ridge(alpha=1).fit(X[tr],target[tr])
    fold_mae.append(mean_absolute_error(target[va],model.predict(X[va])))
print('Validation MAE:',fold_mae)


**Check your result:** Each validation interval follows its training interval.


## Step 3 — Compare final forecasts

The lag-7 baseline predicts the value from the same weekday last week. This is rolling one-step evaluation, not a fixed-origin multi-step forecast.


In [ ]:
model=Ridge(alpha=1).fit(X[:cut],target[:cut])
pred=model.predict(X[cut:])
print('Model MAE:',mean_absolute_error(target[cut:],pred))
print('Seasonal baseline MAE:',mean_absolute_error(target[cut:],X[cut:,1]))
assert np.isfinite(pred).all()


**Check your result:** Both error values are finite.


## Practice

Design a seven-day fixed-origin forecast. Which lag features must be filled with predicted rather than actual values?

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

After the forecast origin, lag-1 often references an unknown future value. Feed back predictions recursively or train a direct model per horizon; evaluate that exact protocol.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
